In [1]:
from google.colab import drive
import os,sys
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
sys.path.append('/content/drive/MyDrive/Github_proj/Deep-Learning-Study/ViT')

In [ ]:
%%writefile /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/model.py

import math
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter,defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import Dataset,Subset,DataLoader,random_split
from tqdm.notebook import tqdm
from torch.optim.lr_scheduler import ReduceLROnPlateau
import itertools
import regex as re
import pickle

Overwriting /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/model.py


# Patch Embed

In [ ]:
%%writefile -a /content/drive/MyDrive/Github_proj/Deep-Learning-Study/ViT/model.py

class PatchEmbed(nn.Module):
    def __init__(self,img_size,patch_size,in_channels=3,embed_dim=768):
      super().__init__()

      assert patch_size > 0
      assert img_size % patch_size == 0

      self.img_size=img_size
      self.patch_size=patch_size
      self.n_patches=(img_size//patch_size)**2

      self.proj=nn.Conv2d(
          in_channels,
          embed_dim,
          kernel_size=patch_size,
          stride=patch_size
      )

    def forward(self,x):
      x=self.proj(x)
      x=x.flatten(2) #(B,D,P,P) -> #(B,D,P^2)
      x=x.transpose(1,2)
      return x


Appending to /content/drive/MyDrive/Github_proj/Deep-Learning-Study/ViT/model.py


# Attention

In [ ]:
%%writefile -a /content/drive/MyDrive/Github_proj/Deep-Learning-Study/ViT/model.py

class MultiHeadAttention(nn.Module):
  def __init__(self, d_model, heads=12, attn_p=0.,proj_p=0.,qkv_bias=True):
    super(MultiHeadAttention,self).__init__()
    assert(d_model % heads == 0)
    self.d_model = d_model
    self.heads = heads
    self.head_dim = d_model // heads
    self.attn_p = attn_p

    self.qkv=nn.Linear(d_model,3*d_model,bias=qkv_bias)
    self.out_fc=nn.Linear(d_model,d_model,bias=qkv_bias)
    self.attn_dropout=nn.Dropout(attn_p)
    self.resid_dropout=nn.Dropout(proj_p)

  def forward(self,x,return_attn=False):
    B, C, D=x.shape
    assert D == self.d_model

    qkv=self.qkv(x).reshape(B,C,3,self.heads,self.head_dim)
    q, k, v=qkv.permute(2,0,3,1,4)

    if return_attn:
      energy=q @ k.transpose(-1,-2)/(self.head_dim**(1/2))
      score=torch.softmax(energy,dim=-1)
      out=self.attn_dropout(score) @ v
    else:
      out= F.scaled_dot_product_attention(
          q,k,v,
          attn_mask=None,
          dropout_p=self.attn_p if self.training else 0.0,
          is_causal=False
      )
      score = None

    out=out.transpose(1,2).reshape(B,C,D)
    out=self.out_fc(out)
    out=self.resid_dropout(out)
    return out,score

Appending to /content/drive/MyDrive/Github_proj/Deep-Learning-Study/ViT/model.py


In [ ]:
%%writefile -a /content/drive/MyDrive/Github_proj/Deep-Learning-Study/ViT/model.py

class FeedForward(nn.Module):
  def __init__(self,d_model,drop_p=0.,bias=True):
    super(FeedForward,self).__init__()
    self.layers=nn.Sequential(
        nn.Linear(d_model,d_model*4,bias=bias),
        nn.GELU(),
        nn.Dropout(drop_p),
        nn.Linear(d_model*4,d_model,bias=bias),
        nn.Dropout(drop_p)
    )

  def forward(self,x):
    x=self.layers(x)
    return x

Appending to /content/drive/MyDrive/Github_proj/Deep-Learning-Study/ViT/model.py


In [ ]:
%%writefile -a /content/drive/MyDrive/Github_proj/Deep-Learning-Study/ViT/model.py

class TransformerBlock(nn.Module):
  def __init__(self,cfg):
    super(TransformerBlock,self).__init__()
    self.norm1=nn.LayerNorm(cfg.d_model,bias=cfg.bias)
    self.attn=MultiHeadAttention(
        d_model=cfg.d_model,
        heads=cfg.heads,
        attn_p=cfg.attn_p,
        proj_p=cfg.proj_p,
        qkv_bias=cfg.bias
    )
    self.norm2=nn.LayerNorm(cfg.d_model,bias=cfg.bias)
    self.mlp=FeedForward(
        d_model=cfg.d_model,
        bias=cfg.bias
    )

  def forward(self,x,return_attn=False):
    attn_out,score=self.attn(self.norm1(x),return_attn)
    x=x+attn_out
    x=x+self.mlp(self.norm2(x))
    return x,score

Appending to /content/drive/MyDrive/Github_proj/Deep-Learning-Study/ViT/model.py


# ViT

In [ ]:
%%writefile -a /content/drive/MyDrive/Github_proj/Deep-Learning-Study/ViT/model.py

class VisionTransformer(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.patch_embed = PatchEmbed(
        img_size=cfg.img_size,
        patch_size=cfg.patch_size,
        in_channels=cfg.in_channels,
        embed_dim=cfg.d_model
    )
    num_patches = (cfg.img_size // cfg.patch_size) ** 2

    self.cls_token = nn.Parameter(torch.zeros(1, 1, cfg.d_model))
    self.pos_embed = nn.Embedding(1 + num_patches, cfg.d_model)
    self.pos_drop = nn.Dropout(cfg.pos_p)

    self.blocks = nn.ModuleList(
        [TransformerBlock(cfg) for _ in range(cfg.n_layers)]
    )
    self.norm = nn.LayerNorm(cfg.d_model, bias=cfg.bias)
    self.head = nn.Linear(cfg.d_model, cfg.n_classes)

    self.apply(self._init_weights)
    nn.init.trunc_normal_(self.cls_token, std=0.02)

  def _init_weights(self, module):
    if isinstance(module, nn.Linear):
      nn.init.trunc_normal_(module.weight, std=0.02)
      if module.bias is not None:
        nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
      nn.init.trunc_normal_(module.weight, std=0.02)
    elif isinstance(module, nn.LayerNorm):
      nn.init.ones_(module.weight)
      if module.bias is not None:
        nn.init.zeros_(module.bias)

  def forward(self, x,return_attn=False):
    x = self.patch_embed(x)
    cls_token = self.cls_token.expand(x.size(0), -1, -1)
    x = torch.cat((cls_token, x), dim=1)
    x=x+self.pos_embed(torch.arange(x.size(1),device=x.device))
    x=self.pos_drop(x)

    scores=[] if return_attn else None
    for block in self.blocks:
      x,score=block(x,return_attn)
      if return_attn:
        scores.append(score)

    x=self.norm(x)
    cls_token_final=x[:,0]
    logits= self.head(cls_token_final)
    return (logits,scores) if return_attn else logits


Appending to /content/drive/MyDrive/Github_proj/Deep-Learning-Study/ViT/model.py


# Smoke Test

In [ ]:
from model import VisionTransformer

In [4]:
class cfg:
  img_size=224
  patch_size=16
  in_channels=3
  n_classes=1000
  d_model=768
  n_layers=12
  heads=12
  bias=True
  pos_p=0.1
  attn_p=0.1
  proj_p=0.1

In [5]:
net = VisionTransformer(cfg)
print(net)

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
  )
  (pos_embed): Embedding(197, 768)
  (pos_drop): Dropout(p=0.1, inplace=False)
  (blocks): ModuleList(
    (0-11): 12 x TransformerBlock(
      (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): MultiHeadAttention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (out_fc): Linear(in_features=768, out_features=768, bias=True)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.0, inplace=False)
          (3): Linear(in_features=3072, out_features=768, bias=True)
          (4): Dropout(p=0.0, inplac

In [26]:
!ls /content/drive/MyDrive/Github_proj/Deep-Learning-Study/ViT
!grep -n "^import\|^from\|%%" /content/drive/MyDrive/Github_proj/Deep-Learning-Study/ViT/model.py

ipynb_file  model.py  __pycache__
2:import math
3:import matplotlib.pyplot as plt
4:import numpy as np
5:from collections import Counter,defaultdict
6:import torch
7:import torch.nn as nn
8:import torch.nn.functional as F
9:import torch.optim as optim
10:import torchvision
11:import torchvision.transforms as transforms
12:import torchvision.datasets as datasets
13:from torch.utils.data import Dataset,Subset,DataLoader,random_split
14:from tqdm.notebook import tqdm
15:from torch.optim.lr_scheduler import ReduceLROnPlateau
16:import itertools
17:import regex as re
18:import pickle


In [8]:
import torch
net.eval()
x = torch.randn(10, 3, 224, 224)
with torch.no_grad():
    fast = net(x)
    slow, scores = net(x, return_attn=True)
print(fast.shape, len(scores), scores[0].shape)
print(torch.allclose(fast, slow, atol=1e-4))

torch.Size([10, 1000]) 12 torch.Size([10, 12, 197, 197])
True


(Img_size/batch_size)^2 + # of cls_token = (224/16)^2 +1 = 197